In [1]:
import os
import re
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")          # non-interactive backend
import seaborn as sns
from collections import Counter

In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

In [3]:
DATA_PATH    = "/content/Good_dataset.csv"
OUTPUT_DIR   = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [4]:
def load_and_explore(path: str) -> pd.DataFrame:
    """Load dataset and print an exploratory summary."""
    df = pd.read_csv(path)
    print("=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)
    print(f"Shape          : {df.shape}")
    print(f"Columns        : {df.columns.tolist()}")
    print(f"\nLabel distribution:\n{df['label'].value_counts()}")
    print(f"\nCategory distribution:\n{df['category'].value_counts()}")
    print(f"\nSeverity distribution:\n{df['severity'].value_counts()}")
    print(f"\nNull values:\n{df.isnull().sum()}")
    print(f"\nSample record:\n{df.iloc[0]}")
    return df

In [5]:
# Patterns compiled once for efficiency
_URL_RE       = re.compile(r"http\S+|www\.\S+")
_HTML_RE      = re.compile(r"<.*?>")
_NON_ALPHA_RE = re.compile(r"[^a-zA-Z\s]")
_MULTI_WS_RE  = re.compile(r"\s+")

# Basic English stop-words (avoids heavy NLTK download)
STOP_WORDS = {
    "i","me","my","myself","we","our","ours","ourselves","you","your","yours",
    "yourself","yourselves","he","him","his","himself","she","her","hers",
    "herself","it","its","itself","they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those","am","is","are",
    "was","were","be","been","being","have","has","had","having","do","does",
    "did","doing","a","an","the","and","but","if","or","because","as","until",
    "while","of","at","by","for","with","about","against","between","into",
    "through","during","before","after","above","below","to","from","up","down",
    "in","out","on","off","over","under","again","further","then","once","here",
    "there","when","where","why","how","all","both","each","few","more","most",
    "other","some","such","no","nor","not","only","own","same","so","than",
    "too","very","s","t","can","will","just","don","should","now","d","ll",
    "m","o","re","ve","y","ain","aren","couldn","didn","doesn","hadn","hasn",
    "haven","isn","ma","mightn","mustn","needn","shan","shouldn","wasn","weren",
    "won","wouldn",
}


def clean_text(text: str) -> str:
    """Full preprocessing: lowercase → strip HTML/URLs → remove punctuation
    → collapse whitespace → remove stop-words."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = _URL_RE.sub(" ", text)
    text = _HTML_RE.sub(" ", text)
    text = _NON_ALPHA_RE.sub(" ", text)
    text = _MULTI_WS_RE.sub(" ", text).strip()
    tokens = [w for w in text.split() if w not in STOP_WORDS and len(w) > 2]
    return " ".join(tokens)


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Apply cleaning and encode labels."""
    df = df.copy()
    df.dropna(subset=["text"], inplace=True)
    df["clean_text"] = df["text"].apply(clean_text)
    # Map FastText-style labels → integers
    df["sentiment_label"] = df["label"].map(
        {"__label__1": 0, "__label__2": 1}
    ).fillna(0).astype(int)

    # Encode complaint category
    le_cat = LabelEncoder()
    df["category_encoded"] = le_cat.fit_transform(df["category"])

    # Severity scores (keyword-based heuristic for demo)
    df["severity_score"] = df["text"].apply(compute_severity_score)

    print("\n[Preprocessing] Done. Cleaned text sample:")
    print(df["clean_text"].iloc[2][:150])
    return df, le_cat



In [6]:
def build_tfidf_features(train_texts, test_texts,
                         max_features: int = 15_000,
                         ngram_range=(1, 2)):
    """Fit TF-IDF on train split, transform both splits."""
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        sublinear_tf=True,   # apply log(1 + tf)
        min_df=2,
    )
    X_train = vectorizer.fit_transform(train_texts)
    X_test  = vectorizer.transform(test_texts)
    print(f"[TF-IDF] vocabulary size: {len(vectorizer.vocabulary_):,}")
    return X_train, X_test, vectorizer


def get_bert_embeddings(texts, batch_size: int = 32, max_length: int = 128):
    """
    Generate sentence embeddings using a pretrained BERT model
    (distilbert-base-uncased for speed; swap for bert-base-uncased for accuracy).

    Returns: np.ndarray of shape (n_samples, hidden_size)
    """
    try:
        from transformers import AutoTokenizer, AutoModel
        import torch

        model_name = "distilbert-base-uncased"
        print(f"[BERT] Loading {model_name} …")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model     = AutoModel.from_pretrained(model_name)
        device    = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device).eval()

        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc   = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            with torch.no_grad():
                out = model(**enc)
            # Mean-pool the last hidden state (CLS pooling is also valid)
            emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(emb)
            if (i // batch_size) % 10 == 0:
                print(f"  … processed {min(i+batch_size, len(texts))}/{len(texts)}")

        embeddings = np.vstack(all_embeddings)
        print(f"[BERT] Embedding shape: {embeddings.shape}")
        return embeddings, tokenizer, model

    except Exception as e:
        print(f"[BERT] Could not generate embeddings: {e}")
        return None, None, None


In [7]:
def train_baseline_model(X_train, y_train, X_test, y_test,
                         target_names, task_name: str = "Sentiment"):
    """Train and evaluate a Logistic Regression baseline."""
    print(f"\n{'='*60}")
    print(f"BASELINE MODEL — Logistic Regression [{task_name}]")
    print("=" * 60)

    clf = LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight="balanced",
        solver="lbfgs",
    )
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average="weighted")
    print(f"\nAccuracy : {acc:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=target_names))

    _plot_confusion_matrix(y_test, preds, target_names,
                           f"Baseline_{task_name}_Confusion_Matrix")
    return clf, acc, f1


def train_category_classifier(X_train, y_train, X_test, y_test, le_cat):
    """Train category classifier (multi-class) on TF-IDF features."""
    categories = le_cat.classes_.tolist()
    return train_baseline_model(
        X_train, y_train, X_test, y_test,
        target_names=categories,
        task_name="Category",
    )

In [8]:
def fine_tune_bert_classifier(train_texts, train_labels,
                               test_texts,  test_labels,
                               num_labels: int = 2,
                               epochs: int = 3,
                               batch_size: int = 16,
                               max_length: int = 128):
    """
    Fine-tune DistilBERT for sequence classification.
    Falls back gracefully when GPU/memory is unavailable.
    """
    try:
        import torch
        from torch.utils.data import Dataset, DataLoader
        from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                                  AdamW, get_linear_schedule_with_warmup)

        class ComplaintDataset(Dataset):
            def __init__(self, texts, labels, tokenizer, max_length):
                self.encodings = tokenizer(
                    list(texts), padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt"
                )
                self.labels = torch.tensor(list(labels), dtype=torch.long)

            def __len__(self):
                return len(self.labels)

            def __getitem__(self, idx):
                item = {k: v[idx] for k, v in self.encodings.items()}
                item["labels"] = self.labels[idx]
                return item

        model_name = "distilbert-base-uncased"
        tokenizer  = AutoTokenizer.from_pretrained(model_name)
        model      = AutoModelForSequenceClassification.from_pretrained(
                         model_name, num_labels=num_labels)
        device     = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)

        train_dataset = ComplaintDataset(train_texts, train_labels, tokenizer, max_length)
        test_dataset  = ComplaintDataset(test_texts,  test_labels,  tokenizer, max_length)
        train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader   = DataLoader(test_dataset,  batch_size=batch_size)

        optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        total_steps = len(train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=total_steps // 10,
            num_training_steps=total_steps,
        )

        print(f"\n{'='*60}")
        print(f"ADVANCED MODEL — Fine-tuning {model_name}")
        print(f"Device: {device}  |  Epochs: {epochs}  |  Batch: {batch_size}")
        print("=" * 60)

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss    = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_loader)
            print(f"  Epoch {epoch+1}/{epochs} — avg loss: {avg_loss:.4f}")

        # Evaluation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch    = {k: v.to(device) for k, v in batch.items()}
                outputs  = model(**batch)
                logits   = outputs.logits
                preds    = torch.argmax(logits, dim=1).cpu().numpy()
                labels   = batch["labels"].cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels)

        acc = accuracy_score(all_labels, all_preds)
        f1  = f1_score(all_labels, all_preds, average="weighted")
        print(f"\n[BERT Fine-tuned] Accuracy: {acc:.4f} | F1: {f1:.4f}")
        print(classification_report(all_labels, all_preds,
                                    target_names=["Negative", "Positive"]))
        return model, tokenizer, acc, f1

    except Exception as e:
        print(f"[BERT Fine-tune] Skipped — {e}")
        return None, None, None, None

In [9]:
POSITIVE_WORDS = {
    "excellent","amazing","great","good","love","best","wonderful","fantastic",
    "outstanding","satisfied","happy","pleased","perfect","recommend","quality",
    "fast","helpful","friendly","efficient","superb"
}
NEGATIVE_WORDS = {
    "terrible","awful","horrible","bad","poor","worst","disappointed","hate",
    "useless","broken","defective","slow","rude","unprofessional","frustrating",
    "waste","scam","refund","damaged","missing","delayed","never","avoid"
}


def rule_based_sentiment(text: str) -> dict:
    """Lexicon-based sentiment scorer."""
    tokens = str(text).lower().split()
    pos = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    if pos > neg:
        label, score = "Positive", round(pos / (pos + neg + 1e-9), 3)
    elif neg > pos:
        label, score = "Negative", round(neg / (pos + neg + 1e-9), 3)
    else:
        label, score = "Neutral", 0.5
    return {"sentiment": label, "pos_count": pos, "neg_count": neg,
            "sentiment_score": score}


def add_sentiment_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich dataframe with rule-based sentiment."""
    sentiment_info = df["text"].apply(rule_based_sentiment).apply(pd.Series)
    return pd.concat([df, sentiment_info], axis=1)

In [10]:
HIGH_SEVERITY_WORDS = {
    "fraud","scam","lawsuit","legal","illegal","dangerous","life","death",
    "urgent","critical","emergency","stolen","hacked","breach","immediately"
}
MEDIUM_SEVERITY_WORDS = {
    "refund","broken","missing","delayed","wrong","error","damaged","lost",
    "cancel","complaint","unacceptable","unresolved","disappointed"
}


def compute_severity_score(text: str) -> int:
    """
    Rule-based severity score.
    Returns 3 (High) / 2 (Medium) / 1 (Low).
    """
    tokens = set(str(text).lower().split())
    if tokens & HIGH_SEVERITY_WORDS:
        return 3
    if tokens & MEDIUM_SEVERITY_WORDS:
        return 2
    return 1


def severity_label(score: int) -> str:
    return {3: "High", 2: "Medium", 1: "Low"}.get(score, "Low")

In [11]:
def trend_analysis(df: pd.DataFrame) -> None:
    """Plot category frequencies, sentiment distribution, and severity breakdown."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Customer Complaint Trend Analysis", fontsize=15, fontweight="bold")

    # 9a — complaint category distribution
    cat_counts = df["category"].value_counts()
    axes[0].barh(cat_counts.index, cat_counts.values,
                 color=sns.color_palette("Blues_r", len(cat_counts)))
    axes[0].set_title("Complaints by Category")
    axes[0].set_xlabel("Count")

    # 9b — sentiment distribution
    if "sentiment" in df.columns:
        sent_counts = df["sentiment"].value_counts()
        colors = ["#ff6b6b","#51cf66","#339af0"][:len(sent_counts)]
        axes[1].pie(sent_counts.values, labels=sent_counts.index,
                    autopct="%1.1f%%", colors=colors, startangle=90)
        axes[1].set_title("Sentiment Distribution")

    # 9c — severity breakdown
    sev_labels = df["severity_score"].map(severity_label).value_counts()
    axes[2].bar(sev_labels.index, sev_labels.values,
                color=["#ff6b6b","#ffd43b","#51cf66"])
    axes[2].set_title("Severity Breakdown")
    axes[2].set_ylabel("Count")

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "trend_analysis.png")
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"\n[Trend Analysis] Chart saved → {out}")


def top_keywords_per_category(df: pd.DataFrame,
                               vectorizer: TfidfVectorizer,
                               le_cat: LabelEncoder,
                               top_n: int = 10) -> None:
    """Print top TF-IDF keywords for each complaint category."""
    print("\n[Top Keywords per Category]")
    for cat_idx, cat_name in enumerate(le_cat.classes_):
        cat_texts = df[df["category_encoded"] == cat_idx]["clean_text"].tolist()
        if not cat_texts:
            continue
        tfidf_matrix = vectorizer.transform(cat_texts)
        mean_tfidf   = tfidf_matrix.mean(axis=0).A1
        top_indices  = mean_tfidf.argsort()[::-1][:top_n]
        terms        = vectorizer.get_feature_names_out()
        keywords     = [terms[i] for i in top_indices]
        print(f"  {cat_name:20s}: {', '.join(keywords)}")

In [12]:
def create_flask_app(sentiment_clf, category_clf,
                     tfidf_vectorizer, le_cat):
    """
    Build a minimal Flask API with three endpoints:
      POST /predict/sentiment   — classify sentiment
      POST /predict/category    — classify complaint category + severity
      POST /predict/full        — all-in-one analysis
    """
    try:
        from flask import Flask, request, jsonify
        app = Flask("ComplaintAnalysisAPI")

        def _vectorize(text):
            cleaned = clean_text(text)
            return tfidf_vectorizer.transform([cleaned])

        @app.route("/health", methods=["GET"])
        def health():
            return jsonify({"status": "ok", "service": "Complaint Analysis API"})

        @app.route("/predict/sentiment", methods=["POST"])
        def predict_sentiment():
            data = request.get_json(force=True)
            text = data.get("text", "")
            if not text:
                return jsonify({"error": "No text provided"}), 400
            features   = _vectorize(text)
            prediction = sentiment_clf.predict(features)[0]
            proba      = sentiment_clf.predict_proba(features).max()
            label      = "Positive" if prediction == 1 else "Negative"
            return jsonify({
                "text"      : text[:200],
                "sentiment" : label,
                "confidence": round(float(proba), 4),
                **rule_based_sentiment(text),
            })

        @app.route("/predict/category", methods=["POST"])
        def predict_category():
            data = request.get_json(force=True)
            text = data.get("text", "")
            if not text:
                return jsonify({"error": "No text provided"}), 400
            features      = _vectorize(text)
            cat_encoded   = category_clf.predict(features)[0]
            category_name = le_cat.inverse_transform([cat_encoded])[0]
            sev_score     = compute_severity_score(text)
            return jsonify({
                "text"         : text[:200],
                "category"     : category_name,
                "severity_score": sev_score,
                "severity_label": severity_label(sev_score),
            })

        @app.route("/predict/full", methods=["POST"])
        def predict_full():
            data = request.get_json(force=True)
            text = data.get("text", "")
            if not text:
                return jsonify({"error": "No text provided"}), 400
            features      = _vectorize(text)

            sent_pred     = sentiment_clf.predict(features)[0]
            sent_proba    = sentiment_clf.predict_proba(features).max()
            cat_encoded   = category_clf.predict(features)[0]
            category_name = le_cat.inverse_transform([cat_encoded])[0]
            sev_score     = compute_severity_score(text)
            rule_sent     = rule_based_sentiment(text)

            return jsonify({
                "text"          : text[:200],
                "sentiment"     : "Positive" if sent_pred == 1 else "Negative",
                "confidence"    : round(float(sent_proba), 4),
                "rule_sentiment": rule_sent["sentiment"],
                "category"      : category_name,
                "severity_score": sev_score,
                "severity_label": severity_label(sev_score),
            })

        return app

    except ImportError:
        print("[API] Flask not available — skipping API creation.")
        return None


# ─────────────────────────────────────────────
# HELPER — confusion matrix plot
# ─────────────────────────────────────────────

def _plot_confusion_matrix(y_true, y_pred, labels, title):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(max(6, len(labels)), max(5, len(labels)-1)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title.replace("_", " "))
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f"{title}.png")
    plt.savefig(out, dpi=130)
    plt.close()
    print(f"  [Plot] Saved → {out}")


In [13]:
def run_pipeline(run_bert: bool = False):
    """
    Execute the full pipeline end-to-end.
    Set run_bert=True to enable BERT fine-tuning (requires GPU / ~15 min CPU).
    """

    # ── 2. Load ──────────────────────────────
    df_raw = load_and_explore(DATA_PATH)

    # ── 3. Preprocess ────────────────────────
    df, le_cat = preprocess_dataframe(df_raw)

    # ── 7. Sentiment enrichment ──────────────
    df = add_sentiment_columns(df)

    # ── Train / test split ───────────────────
    train_df, test_df = train_test_split(
        df, test_size=0.2, random_state=42,
        stratify=df["category_encoded"]
    )
    print(f"\n[Split] Train: {len(train_df):,}  |  Test: {len(test_df):,}")

    # ── 4. TF-IDF features ───────────────────
    X_train_tfidf, X_test_tfidf, vectorizer = build_tfidf_features(
        train_df["clean_text"].tolist(),
        test_df["clean_text"].tolist(),
    )

    # ── 5a. Baseline — Sentiment ─────────────
    sent_clf, sent_acc, sent_f1 = train_baseline_model(
        X_train_tfidf, train_df["sentiment_label"],
        X_test_tfidf,  test_df["sentiment_label"],
        target_names=["Negative", "Positive"],
        task_name="Sentiment",
    )

    # ── 5b. Baseline — Category ──────────────
    cat_clf, cat_acc, cat_f1 = train_category_classifier(
        X_train_tfidf, train_df["category_encoded"],
        X_test_tfidf,  test_df["category_encoded"],
        le_cat,
    )

    # ── 5c. Cross-validation ─────────────────
    print("\n[Cross-Validation] 5-fold on sentiment …")
    cv_scores = cross_val_score(
        LogisticRegression(max_iter=500, class_weight="balanced"),
        X_train_tfidf, train_df["sentiment_label"],
        cv=5, scoring="f1_weighted", n_jobs=-1
    )
    print(f"  CV F1 (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # ── 6. BERT fine-tuning (optional) ───────
    bert_model = bert_tokenizer = None
    if run_bert:
        bert_model, bert_tokenizer, bert_acc, bert_f1 = fine_tune_bert_classifier(
            train_df["clean_text"].tolist(), train_df["sentiment_label"].tolist(),
            test_df["clean_text"].tolist(),  test_df["sentiment_label"].tolist(),
            num_labels=2, epochs=3,
        )

    # ── 9. Trend analysis ────────────────────
    trend_analysis(df)
    top_keywords_per_category(df, vectorizer, le_cat)

    # ── 10. Flask API ─────────────────────────
    app = create_flask_app(sent_clf, cat_clf, vectorizer, le_cat)

    # ── Summary report ───────────────────────
    print("\n" + "=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)
    print(f"  Dataset size          : {len(df):,} records")
    print(f"  Categories            : {le_cat.classes_.tolist()}")
    print(f"  Baseline sentiment    : Accuracy={sent_acc:.4f}  F1={sent_f1:.4f}")
    print(f"  Baseline category     : Accuracy={cat_acc:.4f}  F1={cat_f1:.4f}")
    print(f"  CV F1 (sentiment)     : {cv_scores.mean():.4f}")
    print(f"  Outputs saved to      : {OUTPUT_DIR}")
    if app:
        print(f"\n[API] To start the REST API, call: app.run(port=5000, debug=False)")
    print("=" * 60)

    return {
        "dataframe"       : df,
        "le_cat"          : le_cat,
        "vectorizer"      : vectorizer,
        "sentiment_clf"   : sent_clf,
        "category_clf"    : cat_clf,
        "bert_model"      : bert_model,
        "bert_tokenizer"  : bert_tokenizer,
        "flask_app"       : app,
    }

In [16]:
def run_pipeline(run_bert: bool = False):
    """
    Execute the full pipeline end-to-end.
    Set run_bert=True to enable BERT fine-tuning (requires GPU / ~15 min CPU).
    """

    # ── 2. Load ──────────────────────────────
    df_raw = load_and_explore(DATA_PATH)

    # 🔥 FIX 1: REMOVE original sentiment column EARLY (prevents duplicate + error)
    df_raw = df_raw.drop(columns=['sentiment'], errors='ignore')

    # ── 3. Preprocess ────────────────────────
    df, le_cat = preprocess_dataframe(df_raw)

    # ── 7. Sentiment enrichment (rule-based) ─
    df = add_sentiment_columns(df)

    # ── Train / test split ───────────────────
    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df["category_encoded"]
    )
    print(f"\n[Split] Train: {len(train_df):,}  |  Test: {len(test_df):,}")

    # ── 4. TF-IDF features ───────────────────
    X_train_tfidf, X_test_tfidf, vectorizer = build_tfidf_features(
        train_df["clean_text"].tolist(),
        test_df["clean_text"].tolist(),
    )

    # ── 5a. Baseline — Sentiment ─────────────
    sent_clf, sent_acc, sent_f1 = train_baseline_model(
        X_train_tfidf,
        train_df["sentiment_label"],
        X_test_tfidf,
        test_df["sentiment_label"],
        target_names=["Negative", "Positive"],
        task_name="Sentiment",
    )

    # ── 5b. Baseline — Category ──────────────
    cat_clf, cat_acc, cat_f1 = train_category_classifier(
        X_train_tfidf,
        train_df["category_encoded"],
        X_test_tfidf,
        test_df["category_encoded"],
        le_cat,
    )

    # ── 5c. Cross-validation ─────────────────
    print("\n[Cross-Validation] 5-fold on sentiment …")
    cv_scores = cross_val_score(
        LogisticRegression(max_iter=500, class_weight="balanced"),
        X_train_tfidf,
        train_df["sentiment_label"],
        cv=5,
        scoring="f1_weighted",
        n_jobs=-1
    )
    print(f"  CV F1 (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # ── 6. BERT fine-tuning (optional) ───────
    bert_model = bert_tokenizer = None
    if run_bert:
        bert_model, bert_tokenizer, bert_acc, bert_f1 = fine_tune_bert_classifier(
            train_df["clean_text"].tolist(),
            train_df["sentiment_label"].tolist(),
            test_df["clean_text"].tolist(),
            test_df["sentiment_label"].tolist(),
            num_labels=2,
            epochs=3,
        )

    # ── 9. Trend analysis ────────────────────
    trend_analysis(df)
    top_keywords_per_category(df, vectorizer, le_cat)

    # ── 10. Flask API ─────────────────────────
    app = create_flask_app(sent_clf, cat_clf, vectorizer, le_cat)

    # ── Summary report ───────────────────────
    print("\n" + "=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)
    print(f"  Dataset size          : {len(df):,} records")
    print(f"  Categories            : {le_cat.classes_.tolist()}")
    print(f"  Baseline sentiment    : Accuracy={sent_acc:.4f}  F1={sent_f1:.4f}")
    print(f"  Baseline category     : Accuracy={cat_acc:.4f}  F1={cat_f1:.4f}")
    print(f"  CV F1 (sentiment)     : {cv_scores.mean():.4f}")
    print(f"  Outputs saved to      : {OUTPUT_DIR}")

    if app:
        print(f"\n[API] To start the REST API, call: app.run(port=5000, debug=False)")

    print("=" * 60)

    return {
        "dataframe": df,
        "le_cat": le_cat,
        "vectorizer": vectorizer,
        "sentiment_clf": sent_clf,
        "category_clf": cat_clf,
        "bert_model": bert_model,
        "bert_tokenizer": bert_tokenizer,
        "flask_app": app,
    }

In [17]:
def clean_for_bert(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    return text.strip()

In [19]:
artifacts = run_pipeline(run_bert=False)

DATASET OVERVIEW
Shape          : (9300, 5)
Columns        : ['text', 'label', 'category', 'severity', 'sentiment']

Label distribution:
label
__label__1    7800
__label__2    1500
Name: count, dtype: int64

Category distribution:
category
Delivery Issue      2087
Product Quality     1906
Refund Issue        1888
Customer Service    1712
Payment Issue       1707
Name: count, dtype: int64

Severity distribution:
severity
Low       4166
Medium    4006
High      1128
Name: count, dtype: int64

Null values:
text         0
label        0
category     0
severity     0
sentiment    0
dtype: int64

Sample record:
text         Order 527865 shows delivered but I was home al...
label                                               __label__1
category                                        Delivery Issue
severity                                                  High
sentiment                                             Negative
Name: 0, dtype: object

[Preprocessing] Done. Cleaned text sample:
live 

In [20]:
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
import numpy as np
from sklearn.model_selection import train_test_split

# Ensure artifacts are available from previous pipeline run
# If artifacts is not defined, you might need to run the `run_pipeline` cell first.
if 'artifacts' not in globals():
    print("Error: 'artifacts' not found. Please run the `run_pipeline` cell first.")
else:
    df = artifacts["dataframe"]
    le_cat = artifacts["le_cat"]

    # Re-perform the train/test split to get train_df and test_df
    train_df, test_df = train_test_split(
        df, test_size=0.2, random_state=42,
        stratify=df["category_encoded"]
    )

    # Prepare data
    train_texts = train_df["clean_text"].apply(clean_for_bert).tolist()
    test_texts = test_df["clean_text"].apply(clean_for_bert).tolist()

    train_labels = train_df["sentiment_label"].tolist()
    test_labels = test_df["sentiment_label"].tolist()

    # Convert to HuggingFace Dataset
    train_dataset = Dataset.from_dict({
        "text": train_texts,
        "label": train_labels
    })

    test_dataset = Dataset.from_dict({
        "text": test_texts,
        "label": test_labels
    })

    # Load tokenizer
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=128
        )

    train_dataset = train_dataset.map(tokenize, batched=True)
    test_dataset = test_dataset.map(tokenize, batched=True)

    train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    # Load model
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2
    )


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/7440 [00:00<?, ? examples/s]

Map:   0%|          | 0/1860 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [22]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.000081,1.000000,1.000000
2,0.029369,0.000036,1.000000,1.000000
3,0.000079,0.000029,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 8.110604539979249e-05,
 'eval_accuracy': 1.0,
 'eval_f1': 1.0,
 'eval_runtime': 16.1802,
 'eval_samples_per_second': 114.955,
 'eval_steps_per_second': 7.231,
 'epoch': 3.0}

In [24]:
model.save_pretrained("bert_model")
tokenizer.save_pretrained("bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_model/tokenizer_config.json', 'bert_model/tokenizer.json')

In [25]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    pred = outputs.logits.argmax().item()

    return "Positive" if pred == 1 else "Negative"

In [26]:
import joblib

# Save TF-IDF model
joblib.dump(artifacts["vectorizer"], "vectorizer.pkl")

# Save models
joblib.dump(artifacts["sentiment_clf"], "sentiment_model.pkl")
joblib.dump(artifacts["category_clf"], "category_model.pkl")

# Save label encoder
joblib.dump(artifacts["le_cat"], "label_encoder.pkl")

['label_encoder.pkl']

In [27]:
model.save_pretrained("bert_model")
tokenizer.save_pretrained("bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_model/tokenizer_config.json', 'bert_model/tokenizer.json')

In [28]:
from google.colab import files
files.download("vectorizer.pkl")
files.download("sentiment_model.pkl")
files.download("category_model.pkl")
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>